# Step 1 — Data preparation for the ~110M models (CPU only)

**What this notebook does, in plain words:** it downloads text from several sources (web articles, textbook-style explanations, Wikipedia facts, children's stories), learns one shared **32k-token vocabulary** from all of them, and turns everything into compact token files. It also prepares a separate **question-and-answer set** for fine-tuning later.

The output is used, unchanged, by the Transformer, Mamba and Jamba notebooks, so all three models see exactly the same data.

| Source | Hugging Face | Train tokens | Teaches |
|---|---|---|---|
| FineWeb-Edu | `HuggingFaceFW/fineweb-edu` (`sample-10BT`) | 450M | general knowledge, real writing |
| Cosmopedia v2 | `HuggingFaceTB/smollm-corpus` (`cosmopedia-v2`) | 550M | clear explanations |
| English Wikipedia | `wikimedia/wikipedia` (`20231101.en`) | 250M | facts |
| Simple Wikipedia | `wikimedia/wikipedia` (`20231101.simple`) | up to 200M | facts in easy language |
| TinyStories | `roneneldan/TinyStories` | 150M | coherent stories |
| **Total** | | **~1.6B** | |

**Q&A fine-tuning set** (kept separate, used after pretraining): SQuAD (answer from a passage), Dolly-15k (open questions), smol-smoltalk (short chats).

### How to run on Kaggle
1. *Settings → Accelerator → **None*** (CPU — costs no GPU quota). *Internet → **On***.
2. Optional: *Add-ons → Secrets* → add `HF_TOKEN` (your Hugging Face token) for faster, rate-limit-free downloads.
3. *Save Version → **Save & Run All***. Expect roughly 2–4 hours; the log prints an estimate after the first source.
4. When it finishes: open the version's *Output* → **New Dataset**. Name it e.g. `llm-data-32k`. The training notebooks attach this dataset.

Set the environment variable `SMOKE_TEST=1` to run a tiny offline version with fake data (used to test the code).

## 0. Setup and configuration

In [ ]:
import importlib.util, subprocess, sys
for pkg in ["tokenizers", "datasets"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, json, time, glob, shutil, random, itertools
import numpy as np
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
T_START = time.time()

# optional Hugging Face token from Kaggle secrets (faster downloads, no rate-limit warnings)
if not SMOKE_TEST and not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN loaded from Kaggle secrets")
    except Exception:
        print("No HF_TOKEN secret (optional) — downloads still work, just slower/rate-limited")
print("SMOKE_TEST =", SMOKE_TEST)

In [ ]:
# ---------------- configuration ----------------
OUT = ("/kaggle/working/llm_data" if os.path.isdir("/kaggle/working") else "./llm_data")
if SMOKE_TEST: OUT = "./llm_data_smoke"

VOCAB_SIZE = 32768
SPECIALS = ["<|endoftext|>", "<|user|>", "<|assistant|>", "<|end|>"]
TOKENIZER_SAMPLE_CHARS = 300_000_000      # ~300 MB of text, spread over sources by budget
SHARD_TOKENS = 100_000_000                 # 200 MB per .bin file
TIME_LIMIT_H = 11.0                        # Kaggle sessions end at 12 h
SEED = 1337

# name, HF id, config, how to format a row, train-token budget, val-token budget
SOURCES = [
    dict(name="fineweb_edu", hf="HuggingFaceFW/fineweb-edu",   config="sample-10BT",     fmt="text", budget=450_000_000, val=2_000_000),
    dict(name="cosmopedia",  hf="HuggingFaceTB/smollm-corpus", config="cosmopedia-v2",   fmt="text", budget=550_000_000, val=2_000_000),
    dict(name="wiki_en",     hf="wikimedia/wikipedia",         config="20231101.en",     fmt="wiki", budget=250_000_000, val=1_000_000),
    dict(name="wiki_simple", hf="wikimedia/wikipedia",         config="20231101.simple", fmt="wiki", budget=200_000_000, val=1_000_000),
    dict(name="tinystories", hf="roneneldan/TinyStories",      config=None,              fmt="text", budget=150_000_000, val=1_000_000),
]
SFT_MAX_SMOLTALK = 150_000     # cap the chat set so the fine-tune stays short
SFT_VAL_FRACTION = 0.02

if SMOKE_TEST:
    VOCAB_SIZE, TOKENIZER_SAMPLE_CHARS, SHARD_TOKENS = 1024, 400_000, 20_000
    for s in SOURCES: s["budget"], s["val"] = 30_000, 3_000
    SFT_MAX_SMOLTALK = 200

TOTAL_BUDGET = sum(s["budget"] for s in SOURCES)
for s in SOURCES: s["weight"] = s["budget"] / TOTAL_BUDGET
os.makedirs(OUT, exist_ok=True)
print(f"output: {OUT} | vocab {VOCAB_SIZE} | total train budget {TOTAL_BUDGET/1e9:.2f}B tokens")
for s in SOURCES:
    print(f"  {s['name']:12s} {s['budget']/1e6:8.0f}M train  {s['val']/1e6:5.1f}M val  weight {s['weight']:.2f}")

## 1. Pre-flight checks
Fails fast, before hours of work: is there internet, enough disk, does every dataset/config exist, and does each one contain the fields we expect? It also peeks at one row of each source so you can see what the data looks like.

In [ ]:
# ---------------- data access (real or synthetic) ----------------
def _synthetic_rows(fmt, n=100_000, fail_at=None, seed=0):
    # offline fake data for SMOKE_TEST; `fail_at` simulates a dropped connection once
    rng = random.Random(seed)
    words = ("the a cat dog sun moon river city king queen water fire learn teach science history "
             "number planet energy story friend happy small big run jump because therefore").split()
    for i in range(n):
        if fail_at is not None and i == fail_at and not _synthetic_rows.failed:
            _synthetic_rows.failed = True
            raise ConnectionError("simulated network drop")
        body = " ".join(rng.choice(words) for _ in range(rng.randint(30, 120))).capitalize() + "."
        yield {"title": f"Topic {i}", "text": body} if fmt == "wiki" else {"text": body}
_synthetic_rows.failed = False

def open_stream(src):
    if SMOKE_TEST:
        return _synthetic_rows(src["fmt"], fail_at=250 if src["name"] == "cosmopedia" else None,
                               seed=hash(src["name"]) % 1000)
    from datasets import load_dataset
    return load_dataset(src["hf"], name=src["config"], split="train", streaming=True)

def format_doc(src, row):
    if src["fmt"] == "wiki":
        text = f"{row.get('title', '').strip()}\n\n{row.get('text', '').strip()}"
    else:
        text = (row.get("text") or "").strip()
    return text if len(text) >= 50 else None

def iter_docs(src, max_retries=8):
    # yields formatted documents; on a network error it reopens the stream and skips what was read
    consumed, attempts = 0, 0
    while True:
        try:
            ds = open_stream(src)
            if consumed:
                ds = ds.skip(consumed) if hasattr(ds, "skip") else itertools.islice(ds, consumed, None)
            for row in ds:
                consumed += 1
                doc = format_doc(src, row)
                if doc:
                    yield doc
            return
        except Exception as e:
            attempts += 1
            if attempts > max_retries:
                raise
            wait = min(60, 5 * attempts)
            print(f"  [{src['name']}] stream error after {consumed:,} rows ({type(e).__name__}: {e}); retry {attempts} in {wait}s")
            time.sleep(0 if SMOKE_TEST else wait)

def batched(it, n):
    while True:
        b = list(itertools.islice(it, n))
        if not b: return
        yield b

In [ ]:
# ---------------- pre-flight ----------------
problems = []
try:
    import torch
    if torch.cuda.is_available():
        print("⚠ A GPU is attached but not needed — set Accelerator → None to save GPU quota.")
except ImportError:
    pass

need_gb = TOTAL_BUDGET * 2 / 1e9 * 1.2 + (0 if SMOKE_TEST else 3)
free_gb = shutil.disk_usage(OUT).free / 1e9
if free_gb < need_gb:
    problems.append(f"disk: {free_gb:.1f} GB free, need ~{need_gb:.1f} GB")
if VOCAB_SIZE >= 2**16:
    problems.append("VOCAB_SIZE must be < 65536 (tokens are stored as uint16)")

if not SMOKE_TEST:
    from datasets import get_dataset_config_names
    for s in SOURCES:
        try:
            if s["config"] is not None:
                names = get_dataset_config_names(s["hf"])
                if s["config"] not in names:
                    problems.append(f"{s['hf']}: config '{s['config']}' not found (have e.g. {names[:5]})")
                    continue
        except Exception as e:
            problems.append(f"{s['hf']}: cannot reach dataset — is Internet on? ({type(e).__name__}: {e})")
            continue

for s in SOURCES:
    if any(p.startswith(s["hf"]) for p in problems):
        continue
    try:
        row = next(iter(open_stream(s)))
        need = ["title", "text"] if s["fmt"] == "wiki" else ["text"]
        missing = [f for f in need if f not in row]
        if missing:
            problems.append(f"{s['name']}: missing fields {missing}; row has {list(row)}")
        else:
            print(f"✓ {s['name']:12s} | {format_doc(s, row)[:110]!r}")
    except Exception as e:
        problems.append(f"{s['name']}: could not read a row ({type(e).__name__}: {e})")
_synthetic_rows.failed = False   # the peek must not use up the simulated failure

if problems:
    raise RuntimeError("Pre-flight failed:\n- " + "\n- ".join(problems))
print(f"\nPre-flight OK | disk {free_gb:.0f} GB free (need ~{need_gb:.1f} GB)")

## 2. Q&A fine-tuning set
Every example becomes a short conversation in one shared format:

```
<|user|>
Read the text and answer the question.

Text: ...
Question: Who built the tower?<|end|>
<|assistant|>
Gustave Eiffel's company<|end|>
```

- **SQuAD**: the answer is a span copied from the given passage → teaches *reading comprehension*.
- **Dolly-15k**: human-written answers to open questions (some with a context paragraph) → teaches *answering style*.
- **smol-smoltalk**: short multi-turn chats → teaches *conversation*.

These are saved as JSON lines (`sft/train.jsonl`, `sft/val.jsonl`). Tokenizing and loss-masking happens in the training notebook, so you can change the format later without re-running this notebook.

In [ ]:
def load_sft():
    out = {"squad": [], "dolly": [], "smoltalk": []}
    if SMOKE_TEST:
        for i in range(300):
            out["squad"].append([{"role": "user", "content": f"Read the text and answer the question.\n\nText: The cat {i} sat on the mat.\n\nQuestion: Where did the cat sit?"},
                                 {"role": "assistant", "content": "on the mat"}])
            out["dolly"].append([{"role": "user", "content": f"What is {i} plus one?"}, {"role": "assistant", "content": f"{i+1}."}])
        for i in range(SFT_MAX_SMOLTALK):
            out["smoltalk"].append([{"role": "user", "content": "Hi!"}, {"role": "assistant", "content": f"Hello! How can I help? ({i})"}])
        return out

    from datasets import load_dataset
    for ex in load_dataset("rajpurkar/squad", split="train"):
        if ex["answers"]["text"]:
            out["squad"].append([
                {"role": "user", "content": f"Read the text and answer the question.\n\nText: {ex['context'].strip()}\n\nQuestion: {ex['question'].strip()}"},
                {"role": "assistant", "content": ex["answers"]["text"][0].strip()}])
    for ex in load_dataset("databricks/databricks-dolly-15k", split="train"):
        q = ex["instruction"].strip()
        if ex.get("context", "").strip():
            q += f"\n\nContext: {ex['context'].strip()}"
        if q and ex["response"].strip():
            out["dolly"].append([{"role": "user", "content": q}, {"role": "assistant", "content": ex["response"].strip()}])
    smol = load_dataset("HuggingFaceTB/smol-smoltalk", split="train").shuffle(seed=SEED)
    for ex in smol.select(range(min(SFT_MAX_SMOLTALK, len(smol)))):
        conv = normalize_messages(ex["messages"])
        if conv: out["smoltalk"].append(conv)
    return out

def normalize_messages(msgs):
    # merge system prompts into the first user turn; keep strict user/assistant alternation
    conv, system = [], ""
    for m in msgs:
        role, content = m["role"], (m["content"] or "").strip()
        if role == "system":
            system = content; continue
        if role == "user" and system and not conv:
            content = f"{system}\n\n{content}"; system = ""
        if role not in ("user", "assistant") or not content:
            return None
        expected = "user" if len(conv) % 2 == 0 else "assistant"
        if role != expected:
            return None
        conv.append({"role": role, "content": content})
    if len(conv) < 2 or conv[-1]["role"] != "assistant":
        return None
    return conv

SFT_DIR = os.path.join(OUT, "sft")
if os.path.exists(os.path.join(SFT_DIR, "done.json")):
    sft_counts = json.load(open(os.path.join(SFT_DIR, "done.json")))
    print("SFT set already built:", sft_counts)
else:
    os.makedirs(SFT_DIR, exist_ok=True)
    t0 = time.time()
    sft = load_sft()
    rng = random.Random(SEED)
    train_rows, val_rows = [], []
    for source, convs in sft.items():
        rng.shuffle(convs)
        n_val = max(1, int(len(convs) * SFT_VAL_FRACTION))
        val_rows += [{"source": source, "messages": c} for c in convs[:n_val]]
        train_rows += [{"source": source, "messages": c} for c in convs[n_val:]]
    rng.shuffle(train_rows)
    for name, rows in (("train", train_rows), ("val", val_rows)):
        with open(os.path.join(SFT_DIR, f"{name}.jsonl"), "w") as f:
            for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    sft_counts = {"train": len(train_rows), "val": len(val_rows),
                  "by_source": {k: len(v) for k, v in sft.items()}}
    json.dump(sft_counts, open(os.path.join(SFT_DIR, "done.json"), "w"))
    print(f"SFT set built in {time.time()-t0:.0f}s:", sft_counts)
    del sft, train_rows, val_rows

## 3. Tokenizer: one 32k BPE vocabulary for all sources
**Plain words:** BPE starts from the 256 possible bytes and repeatedly merges the most frequent neighbouring pair into a new token, until there are 32,768 tokens. The sample is drawn from every source in proportion to its budget (plus a little Q&A text), so the vocabulary fits the whole mix — common words, Wikipedia names and story words all get their own tokens.

32k instead of 8k: general text has far more distinct words than TinyStories. With 8k, words like *photosynthesis* would be split into many pieces, wasting context.

In [ ]:
TOK_PATH = os.path.join(OUT, "tokenizer.json")

def sample_for_tokenizer():
    for s in SOURCES:
        quota, n_chars, n_docs = int(TOKENIZER_SAMPLE_CHARS * s["weight"]), 0, 0
        for doc in iter_docs(s):
            yield doc
            n_chars += len(doc); n_docs += 1
            if n_chars >= quota: break
        print(f"  tokenizer sample: {s['name']:12s} {n_docs:,} docs, {n_chars/1e6:.0f}M chars")
    with open(os.path.join(SFT_DIR, "train.jsonl")) as f:      # a little Q&A text
        for line in itertools.islice(f, 20_000):
            for m in json.loads(line)["messages"]:
                yield m["content"]

if os.path.exists(TOK_PATH):
    tok = Tokenizer.from_file(TOK_PATH); print("Tokenizer already trained")
else:
    t0 = time.time()
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIALS, min_frequency=2,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
    tok.train_from_iterator(sample_for_tokenizer(), trainer=trainer)
    tok.save(TOK_PATH)
    print(f"Tokenizer trained in {time.time()-t0:.0f}s")
_synthetic_rows.failed = False

SPECIAL_IDS = {s: tok.token_to_id(s) for s in SPECIALS}
EOT = SPECIAL_IDS["<|endoftext|>"]
assert all(v is not None for v in SPECIAL_IDS.values()), SPECIAL_IDS
assert tok.get_vocab_size() <= VOCAB_SIZE
print("vocab:", tok.get_vocab_size(), "| special ids:", SPECIAL_IDS)

for s in ["The mitochondria is the powerhouse of the cell.", "Once upon a time, Lily found a red ball.",
          "Albert Einstein was born in Ulm, Germany, in 1879.", "<|user|>\nWhat is water?<|end|>"]:
    ids = tok.encode(s).ids
    assert tok.decode(ids, skip_special_tokens=False) == s, "round-trip failed"
    print(f"{len(s):3d} chars -> {len(ids):2d} tokens | {[tok.decode([i], skip_special_tokens=False) for i in ids][:14]}")

## 4. Encode every source into token shards
For each source: the first documents go to `val.bin` until its validation budget is filled, the rest to `train_000.bin`, `train_001.bin`, … (100M tokens each) until the train budget is reached. Each document ends with `<|endoftext|>`.

A source that finished writes a `done.json`; if the notebook is re-run, finished sources are skipped. Network drops are retried automatically.

In [ ]:
def encode_source(src):
    d = os.path.join(OUT, src["name"])
    done_path = os.path.join(d, "done.json")
    if os.path.exists(done_path):
        stats = json.load(open(done_path)); print(f"{src['name']}: already done ({stats['n_train']/1e6:.0f}M)"); return stats
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)

    t0 = time.time()
    val_f = open(os.path.join(d, "val.bin"), "wb")
    shard_idx, shard_n, shard_f = 0, 0, open(os.path.join(d, "train_000.bin"), "wb")
    shards, n_val, n_train, n_docs, n_chars = ["train_000.bin"], 0, 0, 0, 0
    last_print = time.time()
    for docs in batched(iter_docs(src), 2000):
        encs = tok.encode_batch(docs)
        for doc, e in zip(docs, encs):
            arr = np.array(e.ids + [EOT], dtype=np.uint16)
            if n_val < src["val"]:
                arr.tofile(val_f); n_val += len(arr); continue
            if shard_n >= SHARD_TOKENS:
                shard_f.close(); shard_idx += 1; shard_n = 0
                shards.append(f"train_{shard_idx:03d}.bin")
                shard_f = open(os.path.join(d, shards[-1]), "wb")
            arr.tofile(shard_f); shard_n += len(arr); n_train += len(arr)
            n_docs += 1; n_chars += len(doc)
            if n_train >= src["budget"]:
                break
        if n_train >= src["budget"]:
            break
        if time.time() - last_print > 120:
            rate = n_train / (time.time() - t0)
            print(f"  {src['name']}: {n_train/1e6:6.0f}M / {src['budget']/1e6:.0f}M tokens | "
                  f"{rate/1e3:.0f}k tok/s | ETA {(src['budget']-n_train)/max(rate,1)/60:.0f} min")
            last_print = time.time()
        if time.time() - T_START > TIME_LIMIT_H * 3600:
            shard_f.close(); val_f.close()
            raise RuntimeError(f"Time limit reached in {src['name']}. Lower the budgets and re-run "
                               "(finished sources are kept).")
    shard_f.close(); val_f.close()
    if shard_n == 0 and len(shards) > 1:          # drop an empty trailing shard
        os.remove(os.path.join(d, shards.pop()))
    stats = {"n_train": n_train, "n_val": n_val, "shards": shards, "docs": n_docs,
             "chars_per_token": round(n_chars / max(n_train, 1), 2), "seconds": round(time.time() - t0)}
    if n_train < src["budget"]:
        print(f"  note: {src['name']} ran out of data at {n_train/1e6:.0f}M tokens (budget {src['budget']/1e6:.0f}M)")
    json.dump(stats, open(done_path, "w"))
    print(f"✓ {src['name']}: {n_train/1e6:.1f}M train + {n_val/1e6:.2f}M val tokens, {n_docs:,} docs, "
          f"{stats['chars_per_token']} chars/token, {stats['seconds']/60:.1f} min")
    return stats

all_stats = {}
for i, s in enumerate(SOURCES):
    all_stats[s["name"]] = encode_source(s)
    if i == 0 and all_stats[s["name"]]["seconds"] > 0:
        rate = all_stats[s["name"]]["n_train"] / all_stats[s["name"]]["seconds"]
        rest = sum(x["budget"] for x in SOURCES[1:])
        print(f"  → estimated time for remaining sources: {rest/rate/3600:.1f} h")

## 5. Verify and write the manifest
Checks every file: token ids are below the vocab size, sizes match the recorded counts, and a random slice of each source decodes to readable text. `manifest.json` is what the training notebooks read — it lists every file and the mixing weight of each source.

In [ ]:
total_train = sum(st["n_train"] for st in all_stats.values())
manifest = {"vocab_size": VOCAB_SIZE, "tokenizer": "tokenizer.json", "special_tokens": SPECIAL_IDS,
            "chat_format": {"user": "<|user|>\n{content}<|end|>\n", "assistant": "<|assistant|>\n{content}<|end|>\n",
                            "end_of_conversation": "<|endoftext|>"},
            "sources": {}, "sft": {"train": "sft/train.jsonl", "val": "sft/val.jsonl", **sft_counts},
            "total_train_tokens": total_train, "smoke": SMOKE_TEST, "created": time.strftime("%Y-%m-%d %H:%M")}

rng = np.random.default_rng(SEED)
for s in SOURCES:
    st, d = all_stats[s["name"]], os.path.join(OUT, s["name"])
    n_files = 0
    for f in st["shards"] + ["val.bin"]:
        arr = np.memmap(os.path.join(d, f), dtype=np.uint16, mode="r")
        assert arr.max() < tok.get_vocab_size(), f"{s['name']}/{f}: token id out of range"
        n_files += len(arr) if f != "val.bin" else 0
    assert n_files == st["n_train"], f"{s['name']}: shard sizes {n_files} != recorded {st['n_train']}"
    val = np.memmap(os.path.join(d, "val.bin"), dtype=np.uint16, mode="r")
    assert len(val) == st["n_val"]
    first = np.memmap(os.path.join(d, st["shards"][0]), dtype=np.uint16, mode="r")
    i = int(rng.integers(0, max(1, len(first) - 60)))
    print(f"✓ {s['name']:12s} {st['n_train']/1e6:7.1f}M | …{tok.decode(first[i:i+40].tolist())!r}…")
    manifest["sources"][s["name"]] = {"dir": s["name"], "train_shards": st["shards"], "val": "val.bin",
                                      "n_train": st["n_train"], "n_val": st["n_val"],
                                      "weight": round(st["n_train"] / total_train, 4),
                                      "hf": s["hf"], "config": s["config"]}

with open(os.path.join(OUT, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=1)

size_gb = sum(os.path.getsize(p) for p in glob.glob(os.path.join(OUT, "**", "*"), recursive=True) if os.path.isfile(p)) / 1e9
print(f"\nTotal: {total_train/1e9:.3f}B train tokens | {size_gb:.2f} GB on disk | "
      f"SFT {sft_counts['train']:,} train / {sft_counts['val']:,} val conversations")
print("Mixing weights:", {k: v["weight"] for k, v in manifest["sources"].items()})
print(f"Finished in {(time.time()-T_START)/60:.0f} min. Next: Output → New Dataset.")